# Commitment, on TriviaQA (primary) + ontology-trivials (secondary)

`plan_benzon.md` Part 1, final design. Reframed to match `2_nuance.ipynb`'s full-matrix
structure: every follow-up variant tested against every ground truth on every dataset, rather
than pre-assuming which self-report belongs with which ground truth.

**Three ground truths, renamed for brevity (same mechanisms as before):**

- **answer logit** — `metrics.mean_answer_logprob`, the model's own mean answer log-probability
  from the original Phase-0 generation. Free (already populated on every trial regardless of
  dataset). Self-consistency in spirit: how likely is the model to have been "sure" of this
  exact answer.
- **self challenge resistance** and **labeled challenge resistance** — both `commitment_challenge.natural_commitment_challenge_metric`: for
  each trial, whichever evidence argues *against* the model's current answer (the dataset's
  own gold answer if the model was wrong; a counterfeit claim if the model was right), logit
  drop = baseline defend-margin minus challenged defend-margin. The two differ only in where
  the counterfeit claim comes from when one is needed: **self challenge resistance** (Generated-Evidence GT) has
  the model write its own counterfeit claim; **labeled challenge resistance** (Handcrafted-Evidence GT) looks one up
  from a hand-authored table instead (`commitment_challenge.HANDCRAFTED_COUNTERFEITS` for
  TriviaQA, `commitment_challenge.HANDCRAFTED_COUNTERFEITS_ONTOLOGY` for ontology-trivials) —
  added after this notebook's first run showed self challenge resistance correlating only weakly with every
  self-report variant, to check whether that traces to the model's own claim-writing quality
  rather than the challenge mechanism itself. Both run on **both datasets**: TriviaQA's own
  gold-answer aliases give labeled correctness directly; `benzon:ontology_trivials`'s boolean
  `meta["expected_answer"]` is mirrored onto `answers` as literal `("Yes",)`/`("No",)` text
  (`benzon_data.load_ontology_trivials`), and `_model_answered_correctly` detects a single
  Yes/No gold answer and switches to polarity matching (`ground_truth.detect_yes_no_polarity`)
  instead of substring matching, which would false-positive on "No" constantly (it's a
  substring of "cannot", "unknown", "none", ...).

**Four follow-up variants**, in the order the tables below use, same output scale, all tested
against all three ground truths:

1. `commitment` (`sentiment.MACHINE_COMMITMENT`) — "how committed are you to your answer?"
2. `commitment_defined` (`sentiment.MACHINE_COMMITMENT_DEFINED`) — same question as
   `commitment`, prefixed with an explicit dictionary definition of "commitment" — same
   pattern as `2_nuance.ipynb`'s `NUANCE_DEFINED`.
3. `commitment_parallel` (`sentiment.MACHINE_COMMITMENT_PARALLEL`) — "in a parallel timeline, asked
   this same question again, how likely would you give the same answer?"
4. `commitment_challenge` (`sentiment.NATURAL_COMMITMENT`) — "how likely will you change your mind
   if provided evidence?" — named for its ground truth (self challenge resistance/labeled challenge resistance's logit *drop* under an
   evidence challenge), not for its own self-report wording.

**Output scale.** All four now share one unified, 8-level ladder (`sentiment._COMMITMENT_CLASSES`
in `vconf/sentiment.py`) — "Not committed" -> "Barely" -> "Slightly" -> "Somewhat" ->
"Moderately" -> "Mostly" -> "Highly" -> "Fully committed" — replacing what used to be two
differently-worded 4-level ladders (a "commitment" one for the three machine-framed variants,
a separate "persuadable" one for `commitment_challenge`/`NATURAL_COMMITMENT`). Two motivations: the
4-level scale collapsed completely into the top class on some datasets (e.g. all three
machine-framed variants hit 100% "Fully committed" on ontology-trivials earlier in this
notebook's history), leaving no variance to correlate against any ground truth; and comparing
four variants' self-reports side by side in one table reads oddly when two of them use an
entirely different vocabulary for what is, underneath, the same construct.

In [1]:
import json
import os
import pathlib
import sys

import matplotlib
import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

HERE = pathlib.Path.cwd()
with open(HERE / "config.json") as f:
    MODEL_CONFIG = json.load(f)
MODEL_NAME = MODEL_CONFIG["model"]
os.environ["VCONF_MODEL"] = MODEL_NAME
CACHE_DIR = HERE / "cache" / MODEL_NAME
OUT_DIR = HERE / "out" / MODEL_NAME
FIGS_DIR = OUT_DIR / "figs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)
TRIAL_LOG_PATH = CACHE_DIR / "trial.json"
CONSTRUCT_PATH = CACHE_DIR / "construct.json"
SCALE = MODEL_CONFIG.get("scale", {})

from vconf import notebook as nb

_run_config = nb.run_config


def _run_config_with_overrides(base="gemma-categorical", **overrides):
    cfg = _run_config(base, **overrides)
    if MODEL_CONFIG.get("attn_implementation"):
        cfg = cfg.scaled(attn_implementation=MODEL_CONFIG["attn_implementation"])
    return cfg


nb.run_config = _run_config_with_overrides
from vconf import commitment_challenge as CC
from vconf import data as datamod
from vconf import metrics as M
from vconf import pipeline
from vconf.sentiment import (
    MACHINE_COMMITMENT, MACHINE_COMMITMENT_PARALLEL, MACHINE_COMMITMENT_DEFINED, NATURAL_COMMITMENT,
)

cfg = nb.run_config("gemma-categorical", dataset="triviaqa", name="commitment-triviaqa")
print(nb.describe(cfg))

profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : confidence  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


In [2]:
def heatmap(df, vmin=-1, vmax=1):
    """Diverging heatmap styling for a correlation table — blue=positive, red=negative,
    light gray for NaN (a self-report that collapsed to one class, not just weak)."""
    cmap = matplotlib.colormaps["coolwarm"].copy()
    cmap.set_bad(color="#f0f0f0")
    return df.style.format("{:.3f}", na_rep="—").background_gradient(cmap=cmap, vmin=vmin, vmax=vmax, axis=None)

## Dataset

n=100 TriviaQA questions (the paper's own validation split, first 100 by dataset order).

In [3]:
N = SCALE.get("triviaqa", 100)
items = datamod.load_dataset_items("triviaqa", limit=N)
print(f"{len(items)} TriviaQA questions")
pd.DataFrame([{"qid": i.qid, "question": i.question} for i in items]).head()

100 TriviaQA questions

,qid,question
0,tc_2,Who was the man behind The Chipmunks?
1,tc_33,Which Lloyd Webber musical premiered in the US...
2,tc_40,Who was the next British Prime Minister after ...
3,tc_49,Who had a 70s No 1 hit with Kiss You All Over?
4,tc_56,What claimed the life of singer Kathleen Ferrier?


In [4]:
loaded = nb.open_model(cfg, device_map=MODEL_CONFIG.get("device_map"))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Four follow-up variants, one shared Phase-0 pass

In [5]:
VARIANTS = {
    "commitment": MACHINE_COMMITMENT,
    "commitment_defined": MACHINE_COMMITMENT_DEFINED,
    "commitment_parallel": MACHINE_COMMITMENT_PARALLEL,
    "commitment_challenge": NATURAL_COMMITMENT,
}

out = pipeline.run_multi_sentiment(loaded, items, cfg, VARIANTS)
by_qid_variant = {name: {t.qid: t for t in pipeline.filter_valid(trials)} for name, trials in out.items()}
for name, d in by_qid_variant.items():
    print(f"{len(d)}/{len(out[name])} {name} trials valid")

common_qids = sorted(set.intersection(*(set(d) for d in by_qid_variant.values())))
print(f"\n{len(common_qids)} qids valid under all four self-reports")

100/100 commitment trials valid
100/100 commitment_defined trials valid
100/100 commitment_parallel trials valid
100/100 commitment_challenge trials valid

100 qids valid under all four self-reports


## Ground truths: answer logit, self challenge resistance, and labeled challenge resistance

answer logit is free — already populated on every trial regardless of which self-report variant
it's paired with. self challenge resistance and labeled challenge resistance both need the evidence/counterfeit challenge run per trial —
the expensive step (a counterfeit claim for every item the model answered correctly, plus two
forced-choice forward passes per item) — differing only in whether that counterfeit claim is
generated by the model (self challenge resistance) or looked up from the hand-authored table (labeled challenge resistance); the genuine-
evidence branch (item the model got wrong) is identical either way, so the two conditions
share the same `kind` split. All three ground truths depend only on the question/answer, not
on which follow-up phrasing is being tested, so each is computed once and reused across all
four variants below.

In [6]:
any_trial_by_qid = by_qid_variant["commitment"]  # same question/answer across all variants

logit_gt_by_qid = {q: M.mean_answer_logprob(any_trial_by_qid[q]) for q in common_qids}
logit_gt = np.array([logit_gt_by_qid[q] for q in common_qids])

challenge_kinds = {}
gegt_by_qid = {}
hegt_by_qid = {}
for q in common_qids:
    trial = any_trial_by_qid[q]
    challenge = CC.build_natural_commitment_challenge(trial, loaded, cfg)
    challenge_kinds[q] = challenge["kind"]
    gegt_by_qid[q] = CC.natural_commitment_challenge_metric(trial, loaded, cfg, source="generated")
    # genuine-branch trials don't touch the counterfeit claim at all, so self challenge resistance and labeled challenge resistance
    # are identical there — only counterfeit-branch trials need a second, separate pass.
    hegt_by_qid[q] = (
        gegt_by_qid[q] if challenge["kind"] == "genuine"
        else CC.natural_commitment_challenge_metric(trial, loaded, cfg, source="handcrafted")
    )
gegt = np.array([gegt_by_qid[q] for q in common_qids])
hegt = np.array([hegt_by_qid[q] for q in common_qids])

display(pd.Series(challenge_kinds.values()).value_counts().to_frame("count"))
print(f"\nanswer logit mean={logit_gt.mean():.3f} std={logit_gt.std():.3f}")
print(f"self challenge resistance     mean={gegt.mean():.3f} std={gegt.std():.3f}")
print(f"labeled challenge resistance     mean={hegt.mean():.3f} std={hegt.std():.3f}")

,count
genuine,51
counterfeit,49



answer logit mean=-0.178 std=0.185
self challenge resistance     mean=1.745 std=3.861
labeled challenge resistance     mean=1.423 std=3.706


## Full matrix: every follow-up variant × all three ground truths

Every self-report correlated against answer logit, self challenge resistance, and labeled challenge resistance, not just its
originally-intended pairing — the same check the old "cross-exam" section did (does a
self-report track the *other* ground truth just as well as its own, which would mean the two
aren't measuring different things?), now presented as one matrix instead of a hand-picked list
of matched/crossed rows.

In [7]:
selfreport_by_variant = {
    name: np.array([spec.class_midpoint[spec.classes[by_qid_variant[name][q].class_index]] for q in common_qids])
    for name, spec in VARIANTS.items()
}

rho_matrix = pd.DataFrame({
    "answer logit": {name: M.intrinsic_correlation(sr, logit_gt) for name, sr in selfreport_by_variant.items()},
    "self challenge resistance": {name: M.intrinsic_correlation(sr, gegt) for name, sr in selfreport_by_variant.items()},
    "labeled challenge resistance": {name: M.intrinsic_correlation(sr, hegt) for name, sr in selfreport_by_variant.items()},
})
display(heatmap(rho_matrix))

logit_winner = rho_matrix["answer logit"].idxmax()
gegt_winner = rho_matrix["self challenge resistance"].idxmax()
hegt_winner = rho_matrix["labeled challenge resistance"].idxmax()
print(f"winner vs. answer logit : {logit_winner} (rho={rho_matrix.loc[logit_winner, 'answer logit']:.3f})")
print(f"winner vs. self challenge resistance     : {gegt_winner} (rho={rho_matrix.loc[gegt_winner, 'self challenge resistance']:.3f})")
print(f"winner vs. labeled challenge resistance     : {hegt_winner} (rho={rho_matrix.loc[hegt_winner, 'labeled challenge resistance']:.3f})")

,answer logit,self challenge resistance,labeled challenge resistance
commitment,0.356,0.186,0.164
commitment_defined,0.367,0.166,0.102
commitment_parallel,0.320,0.143,0.120
commitment_challenge,0.207,-0.020,-0.113


winner vs. answer logit : commitment_defined (rho=0.367)
winner vs. self challenge resistance     : commitment (rho=0.186)
winner vs. labeled challenge resistance     : commitment (rho=0.164)


## Class distributions

In [8]:
distribution_by_variant = {}
top_share_by_variant = {}
for name, spec in VARIANTS.items():
    dist = M.class_histogram(
        np.array([by_qid_variant[name][q].class_index for q in common_qids]), classes=spec.classes
    )
    distribution_by_variant[name] = dist
    top_share_by_variant[name] = max(dist.values()) / len(common_qids)

display(pd.DataFrame(distribution_by_variant).T)
print("top class share:", {name: f"{s:.1%}" for name, s in top_share_by_variant.items()})
print(f"\nself challenge resistance mean={gegt.mean():.2f} std={gegt.std():.2f}")
print(f"labeled challenge resistance mean={hegt.mean():.2f} std={hegt.std():.2f}")

,Not committed,Barely committed,Slightly committed,Somewhat committed,Moderately committed,Mostly committed,Highly committed,Fully committed
commitment,3,5,0,0,0,0,0,92
commitment_defined,2,6,1,0,0,0,0,91
commitment_parallel,0,7,0,0,0,0,0,93
commitment_challenge,0,4,0,0,0,0,0,96


top class share: {'commitment': '92.0%', 'commitment_defined': '91.0%', 'commitment_parallel': '93.0%', 'commitment_challenge': '96.0%'}

self challenge resistance mean=1.75 std=3.86
labeled challenge resistance mean=1.42 std=3.71


## Validation gate

In [9]:
checks = {
    f"winner vs. answer logit ({logit_winner}) correlates (rho > 0.2)": rho_matrix.loc[logit_winner, "answer logit"] > 0.2,
    f"winner vs. answer logit ({logit_winner}) self-report not collapsed to one class (top class < 95%)":
        top_share_by_variant[logit_winner] < 0.95,
    f"winner vs. self challenge resistance ({gegt_winner}) correlates (rho > 0.2)":
        rho_matrix.loc[gegt_winner, "self challenge resistance"] > 0.2,
    f"winner vs. self challenge resistance ({gegt_winner}) self-report not collapsed to one class (top class < 95%)":
        top_share_by_variant[gegt_winner] < 0.95,
    "self challenge resistance itself shows real variance (std > 0.5)": gegt.std() > 0.5,
    f"winner vs. labeled challenge resistance ({hegt_winner}) correlates (rho > 0.2)":
        rho_matrix.loc[hegt_winner, "labeled challenge resistance"] > 0.2,
    f"winner vs. labeled challenge resistance ({hegt_winner}) self-report not collapsed to one class (top class < 95%)":
        top_share_by_variant[hegt_winner] < 0.95,
    "labeled challenge resistance itself shows real variance (std > 0.5)": hegt.std() > 0.5,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

gate_passed = all(checks.values())
print(f"\n{'GATE PASSED' if gate_passed else 'GATE FAILED'}"
      f" — {'all three ground truths validated on TriviaQA' if gate_passed else 'debug before treating any failing ground truth as validated'}")

PASS  winner vs. answer logit (commitment_defined) correlates (rho > 0.2)
PASS  winner vs. answer logit (commitment_defined) self-report not collapsed to one class (top class < 95%)
FAIL  winner vs. self challenge resistance (commitment) correlates (rho > 0.2)
PASS  winner vs. self challenge resistance (commitment) self-report not collapsed to one class (top class < 95%)
PASS  self challenge resistance itself shows real variance (std > 0.5)
FAIL  winner vs. labeled challenge resistance (commitment) correlates (rho > 0.2)
PASS  winner vs. labeled challenge resistance (commitment) self-report not collapsed to one class (top class < 95%)
PASS  labeled challenge resistance itself shows real variance (std > 0.5)

GATE FAILED — debug before treating any failing ground truth as validated


**Interpretation.** Full matrix above: whichever variant wins each ground truth column is the
wording to keep for that construct going forward — losers can be dropped from `sentiment.py`
once this is confirmed stable, same practice as `2_nuance.ipynb`. self challenge resistance vs. labeled challenge resistance is also
its own comparison: if labeled challenge resistance correlates meaningfully better than self challenge resistance for the same winning
variant, that points at the model's own counterfeit-claim quality as the weak link in the
original challenge GT design, rather than the challenge mechanism or self-report wording. See
the secondary check below for whether the same rankings hold on `benzon:ontology_trivials`.

## Secondary check — all four follow-up variants on the ontology-trivials anchor

`benzon:ontology_trivials` is kept, not discarded (see `vconf/benzon_data.py`), same reason as
`2_nuance.ipynb`. All three ground truths run here now — `load_ontology_trivials` carries
its known yes/no answer as literal `("Yes",)`/`("No",)` gold-answer text (not just
`meta["expected_answer"]`), and `commitment_challenge` has a hand-authored counterfeit table
for this dataset's own qids (`HANDCRAFTED_COUNTERFEITS_ONTOLOGY`), so self challenge resistance and labeled challenge resistance are no
longer TriviaQA-only.

In [10]:
ontology_items = datamod.load_dataset_items("benzon:ontology_trivials", limit=SCALE.get("ontology_trivials"))
print(f"{len(ontology_items)} ontology-trivial questions")
display(pd.Series([item.meta["difficulty"] for item in ontology_items]).value_counts().to_frame("count"))

65 ontology-trivial questions


,count
harder,42
easy,23


In [11]:
cfg_ontology = nb.run_config(
    "gemma-categorical", dataset="benzon:ontology_trivials", name="commitment-ontology-trivials"
)
out_ontology = pipeline.run_multi_sentiment(loaded, ontology_items, cfg_ontology, VARIANTS)
ontology_by_qid_variant = {
    name: {t.qid: t for t in pipeline.filter_valid(trials)} for name, trials in out_ontology.items()
}
for name, d in ontology_by_qid_variant.items():
    print(f"{len(d)}/{len(out_ontology[name])} {name} trials valid")

ontology_common_qids = sorted(set.intersection(*(set(d) for d in ontology_by_qid_variant.values())))
print(f"\n{len(ontology_common_qids)} qids valid under all four self-reports")

ontology_any_trial_by_qid = ontology_by_qid_variant["commitment"]
ontology_logit_gt_by_qid = {q: M.mean_answer_logprob(ontology_any_trial_by_qid[q]) for q in ontology_common_qids}
ontology_logit_gt = np.array([ontology_logit_gt_by_qid[q] for q in ontology_common_qids])

ontology_challenge_kinds = {}
ontology_gegt_by_qid = {}
ontology_hegt_by_qid = {}
for q in ontology_common_qids:
    trial = ontology_any_trial_by_qid[q]
    challenge = CC.build_natural_commitment_challenge(trial, loaded, cfg_ontology)
    ontology_challenge_kinds[q] = challenge["kind"]
    ontology_gegt_by_qid[q] = CC.natural_commitment_challenge_metric(trial, loaded, cfg_ontology, source="generated")
    ontology_hegt_by_qid[q] = (
        ontology_gegt_by_qid[q] if challenge["kind"] == "genuine"
        else CC.natural_commitment_challenge_metric(trial, loaded, cfg_ontology, source="handcrafted")
    )
ontology_gegt = np.array([ontology_gegt_by_qid[q] for q in ontology_common_qids])
ontology_hegt = np.array([ontology_hegt_by_qid[q] for q in ontology_common_qids])

display(pd.Series(ontology_challenge_kinds.values()).value_counts().to_frame("count"))
print(f"\nanswer logit mean={ontology_logit_gt.mean():.3f} std={ontology_logit_gt.std():.3f}")
print(f"self challenge resistance     mean={ontology_gegt.mean():.3f} std={ontology_gegt.std():.3f}")
print(f"labeled challenge resistance     mean={ontology_hegt.mean():.3f} std={ontology_hegt.std():.3f}")

65/65 commitment trials valid
65/65 commitment_defined trials valid
65/65 commitment_parallel trials valid
65/65 commitment_challenge trials valid

65 qids valid under all four self-reports


,count
counterfeit,61
genuine,4



answer logit mean=-0.099 std=0.084
self challenge resistance     mean=2.598 std=2.651
labeled challenge resistance     mean=3.573 std=2.687


In [12]:
ontology_selfreport_by_variant = {
    name: np.array(
        [spec.class_midpoint[spec.classes[ontology_by_qid_variant[name][q].class_index]] for q in ontology_common_qids]
    )
    for name, spec in VARIANTS.items()
}

ontology_rho_matrix = pd.DataFrame({
    "answer logit": {name: M.intrinsic_correlation(sr, ontology_logit_gt) for name, sr in ontology_selfreport_by_variant.items()},
    "self challenge resistance": {name: M.intrinsic_correlation(sr, ontology_gegt) for name, sr in ontology_selfreport_by_variant.items()},
    "labeled challenge resistance": {name: M.intrinsic_correlation(sr, ontology_hegt) for name, sr in ontology_selfreport_by_variant.items()},
})
display(heatmap(ontology_rho_matrix))

ontology_logit_winner = ontology_rho_matrix["answer logit"].idxmax()
ontology_gegt_winner = ontology_rho_matrix["self challenge resistance"].idxmax()
ontology_hegt_winner = ontology_rho_matrix["labeled challenge resistance"].idxmax()
print(f"winner vs. answer logit : {ontology_logit_winner} (rho={ontology_rho_matrix.loc[ontology_logit_winner, 'answer logit']:.3f})")
print(f"winner vs. self challenge resistance     : {ontology_gegt_winner} (rho={ontology_rho_matrix.loc[ontology_gegt_winner, 'self challenge resistance']:.3f})")
print(f"winner vs. labeled challenge resistance     : {ontology_hegt_winner} (rho={ontology_rho_matrix.loc[ontology_hegt_winner, 'labeled challenge resistance']:.3f})")

ontology_distribution_by_variant = {}
ontology_top_share_by_variant = {}
for name, spec in VARIANTS.items():
    dist = M.class_histogram(
        np.array([ontology_by_qid_variant[name][q].class_index for q in ontology_common_qids]), classes=spec.classes
    )
    ontology_distribution_by_variant[name] = dist
    ontology_top_share_by_variant[name] = max(dist.values()) / len(ontology_common_qids)

display(pd.DataFrame(ontology_distribution_by_variant).T)
print("top class share:", {name: f"{s:.1%}" for name, s in ontology_top_share_by_variant.items()})

ontology_checks = {
    f"winner vs. answer logit ({ontology_logit_winner}) correlates (rho > 0.2)":
        ontology_rho_matrix.loc[ontology_logit_winner, "answer logit"] > 0.2,
    f"winner vs. answer logit ({ontology_logit_winner}) self-report not collapsed to one class (top class < 95%)":
        ontology_top_share_by_variant[ontology_logit_winner] < 0.95,
    f"winner vs. self challenge resistance ({ontology_gegt_winner}) correlates (rho > 0.2)":
        ontology_rho_matrix.loc[ontology_gegt_winner, "self challenge resistance"] > 0.2,
    f"winner vs. self challenge resistance ({ontology_gegt_winner}) self-report not collapsed to one class (top class < 95%)":
        ontology_top_share_by_variant[ontology_gegt_winner] < 0.95,
    "self challenge resistance itself shows real variance (std > 0.5)": ontology_gegt.std() > 0.5,
    f"winner vs. labeled challenge resistance ({ontology_hegt_winner}) correlates (rho > 0.2)":
        ontology_rho_matrix.loc[ontology_hegt_winner, "labeled challenge resistance"] > 0.2,
    f"winner vs. labeled challenge resistance ({ontology_hegt_winner}) self-report not collapsed to one class (top class < 95%)":
        ontology_top_share_by_variant[ontology_hegt_winner] < 0.95,
    "labeled challenge resistance itself shows real variance (std > 0.5)": ontology_hegt.std() > 0.5,
}
for name, ok in ontology_checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

ontology_gate_passed = all(ontology_checks.values())
print(f"\n{'PASS' if ontology_gate_passed else 'FAIL'} — all three ground truths "
      f"{'also hold' if ontology_gate_passed else 'do not all hold'} on the ontology-trivials anchor")

/home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/vconf/metrics.py:202: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(stats.spearmanr(x[mask], y[mask]).correlation)


,answer logit,self challenge resistance,labeled challenge resistance
commitment,—,—,—
commitment_defined,—,—,—
commitment_parallel,—,—,—
commitment_challenge,0.113,-0.187,-0.013


winner vs. answer logit : commitment_challenge (rho=0.113)
winner vs. self challenge resistance     : commitment_challenge (rho=-0.187)
winner vs. labeled challenge resistance     : commitment_challenge (rho=-0.013)


,Not committed,Barely committed,Slightly committed,Somewhat committed,Moderately committed,Mostly committed,Highly committed,Fully committed
commitment,0,0,0,0,0,0,0,65
commitment_defined,0,0,0,0,0,0,0,65
commitment_parallel,0,0,0,0,0,0,0,65
commitment_challenge,0,1,0,0,0,0,0,64


top class share: {'commitment': '100.0%', 'commitment_defined': '100.0%', 'commitment_parallel': '100.0%', 'commitment_challenge': '98.5%'}
FAIL  winner vs. answer logit (commitment_challenge) correlates (rho > 0.2)
FAIL  winner vs. answer logit (commitment_challenge) self-report not collapsed to one class (top class < 95%)
FAIL  winner vs. self challenge resistance (commitment_challenge) correlates (rho > 0.2)
FAIL  winner vs. self challenge resistance (commitment_challenge) self-report not collapsed to one class (top class < 95%)
PASS  self challenge resistance itself shows real variance (std > 0.5)
FAIL  winner vs. labeled challenge resistance (commitment_challenge) correlates (rho > 0.2)
FAIL  winner vs. labeled challenge resistance (commitment_challenge) self-report not collapsed to one class (top class < 95%)
PASS  labeled challenge resistance itself shows real variance (std > 0.5)

FAIL — all three ground truths do not all hold on the ontology-trivials anchor


**Interpretation.** This secondary check corroborates (or flags a mismatch with) the
TriviaQA-based winners above using a completely different, difficulty-tiered dataset, now
across all three ground truths. The TriviaQA matrix remains the one that decides whether
Phase 3 (Synonyms) proceeds; a mismatch here is worth a closer look, not an automatic
blocker.

## Full matrix: every follow-up variant × {answer logit, self challenge resistance, labeled challenge resistance} × {TriviaQA, ontology-trivials}

Everything above in one table — `NaN` marks a self-report that completely collapsed to one
class on that dataset (Spearman rho undefined for a constant input, not just weak).

In [13]:
full_matrix = pd.DataFrame({
    "TriviaQA x answer logit": rho_matrix["answer logit"],
    "TriviaQA x self challenge resistance": rho_matrix["self challenge resistance"],
    "TriviaQA x labeled challenge resistance": rho_matrix["labeled challenge resistance"],
    "ontology x answer logit": ontology_rho_matrix["answer logit"],
    "ontology x self challenge resistance": ontology_rho_matrix["self challenge resistance"],
    "ontology x labeled challenge resistance": ontology_rho_matrix["labeled challenge resistance"],
}).reindex(VARIANTS.keys())
full_matrix.index.name = "follow-up variant"

display(heatmap(full_matrix))

,TriviaQA x answer logit,TriviaQA x self challenge resistance,TriviaQA x labeled challenge resistance,ontology x answer logit,ontology x self challenge resistance,ontology x labeled challenge resistance
follow-up variant,,,,,,
commitment,0.356,0.186,0.164,—,—,—
commitment_defined,0.367,0.166,0.102,—,—,—
commitment_parallel,0.320,0.143,0.120,—,—,—
commitment_challenge,0.207,-0.020,-0.113,0.113,-0.187,-0.013


## Log every cell to `trial.json`

Persist every rho this notebook just computed to the shared calibration log
(`vconf.trial_log`) — `conclusion.ipynb` reads this back to know which
`(dataset, OM, GT)` cells already exist and which are still missing.

In [14]:
from vconf import trial_log as TL

records = []
for dataset_name, matrix, n in [
    ("triviaqa", rho_matrix, len(common_qids)),
    ("ontology_trivials", ontology_rho_matrix, len(ontology_common_qids)),
]:
    for om in matrix.index:
        for gt_display, gt_key in TL.GT_KEY.items():
            if gt_display in matrix.columns:
                records.append({"dataset": dataset_name, "om": om, "gt": gt_key, "rho": matrix.loc[om, gt_display], "n": n})

TL.upsert(records, path=TRIAL_LOG_PATH)
print(f"logged {len(records)} cells to {TRIAL_LOG_PATH}")

logged 24 cells to /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/phase_0_calibration/cache/qwen/trial.json
